# 第五章（二）：变量关系、变化与探索性分析表达

**授课教师：** 王润祖

In [ ]:
import os
from pathlib import Path

from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

BASE_DIR = Path(".")
if not (BASE_DIR / "data" / "student_eda.csv").exists():
    BASE_DIR = Path("lecture/chapter5")

font_path = BASE_DIR / "assets" / "NotoSansCJKsc-Regular.otf"
if font_path.exists():
    font_manager.fontManager.addfont(str(font_path))
    cjk_font = font_manager.FontProperties(fname=str(font_path)).get_name()
    plt.rcParams["font.family"] = cjk_font
    plt.rcParams["axes.unicode_minus"] = False

major_order = ["统计学", "应用统计学", "数据科学"]
student_df = pd.read_csv(BASE_DIR / "data" / "student_eda.csv")
wave_df = pd.read_csv(BASE_DIR / "data" / "learning_wave.csv")
student_df["major"] = pd.Categorical(student_df["major"], categories=major_order, ordered=True)
wave_df["major"] = pd.Categorical(wave_df["major"], categories=major_order, ordered=True)
student_df.head()

## 本节概览

* 用散点图探索两个数值变量的关系
* 使用 `hue` 引入第三个分类变量
* 先分组汇总，再绘制有序记录的变化图
* 读懂相关矩阵热力图，并理解相关不等于因果

# 从变量关系开始

## 两个数值变量可以先画散点图

* 散点图适合观察两个数值变量之间是否存在共同变化。
* 每个点通常对应一条记录。
* 关注整体趋势、离散程度、可能的非线性和异常点。

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

sns.scatterplot(
    data=student_df,
    x="study_hours",
    y="python_score",
    ax=ax,
)
ax.set(
    title="学习时长与 Python 成绩",
    xlabel="每周学习时长（小时）",
    ylabel="Python 成绩",
)

fig.tight_layout()
plt.show()

## `hue`：把第三个变量编码为颜色

* `hue` 可以把一个分类变量编码为颜色。
* 颜色用于比较组别，不应用来代替完整的分组统计。
* 类别太多时，颜色会难以区分，应先考虑是否真的需要全部展示。

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

sns.scatterplot(
    data=student_df,
    x="study_hours",
    y="python_score",
    hue="major",
    hue_order=major_order,
    ax=ax,
)
ax.set(
    title="不同专业的学习时长与 Python 成绩",
    xlabel="每周学习时长（小时）",
    ylabel="Python 成绩",
)
ax.legend(title="专业")

fig.tight_layout()
plt.show()

## 小结任务 1

绘制 `attendance_rate` 与 `python_score` 的散点图，并按 `gender` 着色。写出一个图形支持的观察，以及一个图形不能直接回答的问题。

In [ ]:
# fig, ax = plt.subplots(figsize=(7, 4))
# sns.scatterplot(
#     data=student_df,
#     x=...,
#     y=...,
#     hue=...,
#     ax=ax,
# )
# ax.set(title=..., xlabel=..., ylabel=...)
# ax.legend(title=...)
# fig.tight_layout()
# plt.show()

请根据前面的散点图，写出一个图形支持的观察，以及一个图形不能直接回答的问题。

# 描述性趋势线的边界

## `sns.regplot()`：一条趋势线能说明什么

* `regplot()` 可以在散点图上叠加线性趋势线。
* 它适合帮助观察总体线性变化方向。
* 趋势线不自动证明因果关系，也不替代回归诊断和统计推断。

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

sns.regplot(
    data=student_df,
    x="study_hours",
    y="python_score",
    ci=None,
    scatter_kws={"alpha": 0.6},
    line_kws={"color": "#E15759"},
    ax=ax,
)
ax.set(
    title="学习时长与 Python 成绩的描述性趋势",
    xlabel="每周学习时长（小时）",
    ylabel="Python 成绩",
)

fig.tight_layout()
plt.show()

### 图形观察与统计推断的边界

从图上看到正向趋势，可以写“在当前样本中，学习时长较高的学生往往也有较高的 Python 成绩”。不能仅凭这张图写“增加学习时长一定会提高成绩”。回归系数、残差、拟合程度和因果解释边界将在下一章进一步讨论。

# 有序记录与变化图

## 先把多轮记录汇总清楚

* 多轮数据中，每位学生会出现多次。
* 作图前先明确每个点代表什么：个人记录、专业均值，还是全体学生均值。
* 本节先使用第四章的 `groupby()` 生成按轮次、按专业的平均成绩。

In [ ]:
wave_summary = (
    wave_df
    .groupby(["wave", "major"], observed=True)
    .agg(
        mean_score=("python_score", "mean"),
        n=("student_id", "count"),
    )
    .reset_index()
)

print(wave_summary.head(9))

## `sns.lineplot()`：展示有序变化

* 折线图适合横轴有自然顺序的场景，例如时间、轮次或测量阶段。
* 如果横轴只是无序类别，不应把类别硬连成折线。
* 这里的每个点都是一个专业在某一轮的平均成绩，而不是单个学生。

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

sns.lineplot(
    data=wave_summary,
    x="wave",
    y="mean_score",
    hue="major",
    hue_order=major_order,
    marker="o",
    ax=ax,
)
ax.set(
    title="不同专业的平均 Python 成绩变化",
    xlabel="学习轮次",
    ylabel="平均 Python 成绩",
)
ax.legend(title="专业")

fig.tight_layout()
plt.show()

## 快速辨析：有类别不等于要连线

下面左图故意把无自然顺序的专业类别连成折线。它会暗示“专业一、专业二、专业三”之间存在连续变化，但专业只是分类标签；右图用柱形图比较同一组均值，更符合这个问题。折线图应留给 `wave` 这样的轮次或时间顺序。

这里只需要根据横轴是否具有自然顺序选择图形。还要记住：无论使用柱形图、折线图还是散点图，图形中的组间差异和共同变化都不能单独证明因果关系。

In [ ]:
major_score_summary = (
    student_df
    .groupby("major", observed=True)
    .agg(mean_score=("python_score", "mean"))
    .reindex(major_order)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# 不规范示例：把无序类别当成连续横轴。
sns.lineplot(
    data=major_score_summary,
    x="major",
    y="mean_score",
    marker="o",
    color="#E15759",
    errorbar=None,
    ax=axes[0],
)
axes[0].set(title="不规范示例：无序类别连线", xlabel="专业", ylabel="平均 Python 成绩")

# 修正示例：类别比较使用柱形图。
sns.barplot(
    data=major_score_summary,
    x="major",
    y="mean_score",
    color="#59A14F",
    errorbar=None,
    ax=axes[1],
)
axes[1].set(title="修正示例：类别用柱形比较", xlabel="专业", ylabel="平均 Python 成绩")

fig.tight_layout()
plt.show()

## 小结任务 2

按 `wave` 汇总总体平均学习时长，并绘制折线图。说明横轴为什么适合使用折线图。

In [ ]:
# hour_summary = (
#     wave_df
#     .groupby(...)
#     .agg(mean_hours=(..., ...))
#     .reset_index()
# )
#
# fig, ax = plt.subplots(figsize=(6, 4))
# sns.lineplot(data=hour_summary, x=..., y=..., marker="o", ax=ax)
# ax.set(title=..., xlabel=..., ylabel=...)
# fig.tight_layout()
# plt.show()

请解释为什么 `wave` 适合放在折线图横轴上，并简要描述平均学习时长的变化。

# 相关矩阵与热力图

## 相关矩阵

* `corr()` 计算数值变量之间的线性相关系数矩阵。
* 正相关大致表示两个变量倾向于同升同降，负相关大致表示一个变量高时另一个变量低。
* 相关系数接近 `1` 或 `-1` 表示线性关系较强，接近 `0` 表示线性关系较弱。
* 相关矩阵只针对数值变量；分类变量不应直接放入其中。

In [ ]:
numeric_cols = [
    "study_hours",
    "attendance_rate",
    "python_score",
    "data_literacy_score",
]

corr = student_df[numeric_cols].corr()
print(corr.round(2))

## `sns.heatmap()`：把矩阵变成颜色图

* 热力图把矩阵中的数值映射为颜色，适合快速比较结构。
* `annot=True` 显示具体数值，`fmt=".2f"` 控制小数位。
* 相关热力图的颜色深浅不代表因果强度。

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    center=0,
    vmax=1,
    ax=ax,
)
ax.set(title="学习行为与成绩指标的相关矩阵")

fig.tight_layout()
plt.show()

## 小结任务 3

对 `student_df` 中的数值列计算相关矩阵，并绘制热力图。选择其中一对变量，写出相关方向和一个合理的解释边界。

In [ ]:
# selected_cols = [...]
# corr = student_df[selected_cols].corr()
#
# fig, ax = plt.subplots(figsize=(7, 5))
# sns.heatmap(
#     corr,
#     annot=True,
#     fmt=...,
#     cmap=...,
#     vmin=...,
#     vmax=...,
#     ax=ax,
# )
# ax.set(title=...)
# fig.tight_layout()
# plt.show()

请从相关矩阵中选择一对变量，说明相关方向，并写出一个合理的解释边界。

# 从图形到 EDA 表达

## 一组图回答一组问题

一个小型 EDA 不需要画很多图，而应让每张图承担不同任务：

1. 分布图：样本整体是什么样。
2. 分组图：哪些组值得比较。
3. 关系图或变化图：变量之间有哪些值得进一步分析的结构。

### 结果表达的基本顺序

1. 描述图上直接看到的现象。
2. 说明样本、指标或图形的限制。
3. 提出可以进一步检验的问题。

避免把相关、组间差异或时间变化直接写成因果结论。

## 综合案例

问题：模拟样本中，学习投入、出勤与 Python 成绩之间有哪些值得进一步分析的结构？

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

sns.scatterplot(
    data=student_df,
    x="attendance_rate",
    y="python_score",
    hue="major",
    hue_order=major_order,
    alpha=0.75,
    ax=ax,
)
ax.set(
    title="出勤率与 Python 成绩",
    xlabel="出勤率（%）",
    ylabel="Python 成绩",
)
ax.legend(title="专业")

fig.tight_layout()
plt.show()

### 一种合格的 EDA 表达

模拟样本中，出勤率与 Python 成绩呈现正向共同变化；不同专业的点云位置也有一定差别。由于这里没有控制学生基础、课程难度等其他因素，图形只能说明样本中的关联结构，不能单独证明出勤率造成了成绩差异。

## 课堂练习

围绕 `student_df` 和 `wave_df` 完成一个三图 EDA：

1. 用一张分布或分组图描述 Python 成绩。
2. 用散点图探索学习时长或出勤率与成绩的关系。
3. 用折线图或相关热力图展示进一步的结构。
4. 用 100 字以内写出：一个图形事实、一个解释边界、一个可进一步检验的问题。

In [ ]:
# 图 1：分布或分组差异

# 图 2：变量关系

# 图 3：变化或相关结构

请用 100 字以内概括三图 EDA：写出一个图形事实、一个解释边界和一个可进一步检验的问题。

## 本节小结

* 散点图用于探索数值变量关系，`hue` 可引入一个分类变量。
* 趋势线帮助描述总体方向，但不自动给出因果结论。
* 多轮数据先用 `groupby()` 汇总，再使用折线图表达有序变化。
* 相关热力图帮助比较数值变量之间的线性结构，相关不等于因果。
* EDA 的目标是提出清楚、可继续检验的问题，而不是只生成图形。

## 下一章

下一章将从图形观察进入两组比较和简单线性回归，并练习规范解释统计结果。